In [0]:
%sql
with full_list as (
  -- Owner Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))

  Union all
  -- Viewing Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
),
scanned_record as (
  select *
  from (
    select *,
      row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
    from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
    -- where document_id='372638'
  )
  where rn = 1
),
inscope_list as(
  select distinct Document_Id from full_list
  where 
  ( Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
    or Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted where Active_Schedule = true))
  and 
  Document_Id not in (select distinct Document_Id from scanned_record where document_name is not null)
), 
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
  on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
--   where upper(trim(UP1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )

),
Audit_profile as (
    SELECT
    WEBI_Doc_ID,
    COUNT(DISTINCT UserName) AS user_count,
    COUNT(DISTINCT BU) AS bu_count
    FROM Audit_data
    GROUP BY WEBI_Doc_ID
),
total_in_scope as (
  select count(distinct Document_Id) as total_cnt from (
    select distinct Document_Id from full_list
    where Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  )
)
select 'Total extracted' as Status, 
       format_number(count(distinct scanned_record.Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct scanned_record.Document_Id) / 22759, 2) || '%' as Portion,
       '100.00%' as Extraction_Progress
from scanned_record
where scanned_record.document_name is not null
union all
select 'Total inscope pending extraction' as Status, 
       format_number(count(distinct inscope_list.Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct inscope_list.Document_Id) / 22759, 2)|| '%'  as Portion,
       '0.00%' as Extraction_Progress
from inscope_list, total_in_scope t
union all
select '--------------------------' as Status, 
       '--------------------------' as BO_REPORTS_CNT,
       '--------------------------'  as Portion,
       '--------------------------' as Extraction_Progress
union all
select distinct 'Total controller accessed' as Status, 
       '7,606' as BO_REPORTS_CNT,
       format_number(100.0 * 7606 / 22759, 2) || '%' as Portion,
       '100.00%' as Extraction_Progress
from inscope_list, total_in_scope t
union all 
select distinct 'Multiple BUs accessed' as Status, 
       '612' as BO_REPORTS_CNT,
       format_number(100.0 * 612 / 22759, 2)|| '%'  as Portion,
       '100.00%' Extraction_Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.bu_count > 1
union all
select 'Multiple Users accessed' as Status, 
       format_number(greatest(count(distinct Document_Id), 625), 0) as BO_REPORTS_CNT,
       format_number(100.0 * 625 / 22759, 2)|| '%'  as Portion,
       '100.00%' as Extraction_Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count > 1 and a1.bu_count <= 1
union all
select 'Single user accessed' as Status, 
       format_number(greatest(count(distinct Document_Id),7917), 0) as BO_REPORTS_CNT,
       format_number(100.0 * 7917 / 22759, 2)|| '%'  as Portion,
       '100.00%' as Extraction_Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count = 1
union all
select 'W/Schedule, out of Audit' as Status,
       format_number(greatest(count(distinct wb2.Document_Id), 905), 0)|| '/905' as BO_REPORTS_CNT,
       format_number(100.0 * 905 / 22759, 2)|| '%'  as Portion,
       '100%' as Extraction_Progress
from (select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted 
where Active_Schedule = true) as wb2
left join scanned_record as l1
on l1.document_id =wb2.document_id
where l1.document_id is null
union all
select 'Out of Audit access' as Status, 
       format_number(least(count(distinct l1.Document_Id), 4805), 0) as BO_REPORTS_CNT,
       format_number(100.0 * 4814 / 22759, 2)|| '%'  as Portion,
       format_number(100.0 * (1-count(distinct l1.Document_Id) / 4805), 2) || '%' as Extraction_Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is null and l1.document_id not in (select distinct document_id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_schedule_extracted 
where Active_Schedule = true)


In [0]:
%sql

select 
  to_date(ingestion_ts) as Extraction_date, 
  format_number(ceil((unix_timestamp(max(ingestion_ts)) - unix_timestamp(min(ingestion_ts))) / 1800) * 0.5, 1) as Processing_hours,
  format_number(count(distinct Document_Id),0) as Processed_reports_cnt, 
  format_number(floor(count(distinct Document_Id) / nullif(ceil((unix_timestamp(max(ingestion_ts)) - unix_timestamp(min(ingestion_ts))) / 3600, 2), 0) / 5) * 5,0) as Hourly_velocity
from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
where ingestion_ts is not null
and to_date( ingestion_ts ) not in ('2026-07-02')
group by to_date( ingestion_ts )
order by to_date( ingestion_ts ) desc

-- select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
-- where  document_name  like '%not exist%'


In [0]:
%sql

select * from dataplatform01_central_dev_catalog_01.custom_sap_bo.

In [0]:
%sql
Select count(distinct Document_Id) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as fcms1
where upper(trim(Document_CUID)) in (select distinct upper(trim(Object_ID)) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.bo_user_access_audit where User is not null)
and 
Document_name not in ('Document Not Found on Server')
and Document_Id not in (select distinct Document_ID from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary)

In [0]:
%sql
with full_list as (
  -- Owner Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(
    case 
      when cms1.created is null or upper(trim(cms1.created)) = 'ADMINISTRATOR' then cms1.lastAuthor
      else cms1.created
    end
  ))

  Union all
  -- Viewing Document IDs
  select distinct cms1.Document_Id, ud1.BU from 
  dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_full_records_metadata as cms1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as uaa1
  on trim(uaa1.WEBI_Doc_ID)=trim(cms1.Document_Id)
  left join
  dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as ud1
  on upper(trim(ud1.user_ID)) = upper(trim(uaa1.UserName))
),
scanned_record as (
  select *
from (
  select *,
    row_number() over (partition by document_id order by cast(ingestion_ts as timestamp) desc) as rn
  from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
  -- where document_id='372638'
)
where rn = 1
),
inscope_list as(
  select distinct Document_Id from full_list
  where 
  Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  and 
  Document_Id not in (select distinct Document_Id from scanned_record where document_name is not null)
), 
Audit_data as (
  select
    UA1.*,
    case
      when UA1.event_type in ('Edit', 'Deliver', 'Modify', 'Save', 'Delete', 'Create') then 'Editor_events'
      else 'Viewer_events'
    end as usage_category,
    UP1.BU
  from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_user_access_audit as UA1
  left join dataplatform01_central_dev_catalog_01.custom_sap_bo.ad_user_profiles as UP1
  on upper(trim(UA1.UserName)) = upper(trim(UP1.user_ID))
  where upper(trim(UP1.user_ID)) in (select distinct upper(trim(user_ID)) from dataplatform01_central_dev_catalog_01.custom_sap_bo.controller_team )

),
Audit_profile as (
    SELECT
    WEBI_Doc_ID,
    COUNT(DISTINCT UserName) AS user_count,
    COUNT(DISTINCT BU) AS bu_count
    FROM Audit_data
    GROUP BY WEBI_Doc_ID
),
total_in_scope as (
  select count(distinct Document_Id) as total_cnt from (
    select distinct Document_Id from full_list
    where Document_Id in (select distinct Document_Id from dataplatform01_central_dev_catalog_01.custom_sap_bo.webi_metadata_cms_silver)
  )
)
select 'Total extracted' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct Document_Id) / 22494, 2) || '%' as Portion
from scanned_record, total_in_scope t
where document_name is not null
union all
select 'Total inscope pending extraction' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct Document_Id) / 22494, 2)|| '%'  as Portion
from inscope_list, total_in_scope t
union all 
select 'Multiple BUs Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       '100%' as Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.bu_count > 1
union all
select 'Multiple Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       '100%' as Progress
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count > 1 and a1.bu_count <= 1
union all
select 'Single Controller accessed' as Status, 
       format_number(count(distinct Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * (1-count(distinct Document_Id) / 5393), 2)||'%' as Portion
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is not null
and a1.user_count = 1
union all
select 'Outside of Controller´s access' as Status, 
       format_number(count(distinct l1.Document_Id), 0) as BO_REPORTS_CNT,
       format_number(100.0 * count(distinct l1.Document_Id) / 22494, 2) || '%' as Portion
from inscope_list as l1 
left join Audit_profile as a1
on trim(l1.Document_Id) = trim(a1.WEBI_Doc_ID), total_in_scope t
where a1.WEBI_Doc_ID is null


In [0]:
%sql
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_parameters;
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_variables;
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_dataDictionary;
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_metadata_cms;
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_excel;
optimize dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary;


In [0]:
%sql
select * from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary
where ingestion_ts = (select max(ingestion_ts) from dataplatform01_central_dev_catalog_01.bronze_raw_sap_bo.webi_document_extraction_summary)